In [2]:
import pandas as pd
import numpy as np

In [ ]:
import numpy as np

In [4]:
!pip install -q pypdf
from pypdf import PdfReader

# Read the PDF file
reader = PdfReader("/content/Buy Computer.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print("Extracted Text:")
print(text[:1000]) # Display first 1000 characters

Extracted Text:
age income student credit_rating buys_computer
<=30 high no fair no
<=30 high no excellent no
31...40 high no fair yes
>40 medium no fair yes
>40 low yes fair yes
>40 low yes excellent no
31...40 low yes excellent yes
<=30 medium no fair no
<=30 low yes fair yes
>40 medium yes fair yes
<=30 medium yes excellent yes
31...40 medium no excellent yes
31...40 high yes fair yes
>40 medium no excellent no



In [8]:
import io

# Parse the extracted text into a DataFrame using a raw string for the regex separator
df = pd.read_csv(io.StringIO(text), sep=r'\s+')

# Replace the age range format and display the DataFrame
df['age'] = df['age'].str.replace('31...40', '31-40', regex=False)
df

,age,income,student,credit_rating,buys_computer
0,<=30,high,no,fair,no
1,<=30,high,no,excellent,no
2,31-40,high,no,fair,yes
3,>40,medium,no,fair,yes
4,>40,low,yes,fair,yes
5,>40,low,yes,excellent,no
6,31-40,low,yes,excellent,yes
7,<=30,medium,no,fair,no
8,<=30,low,yes,fair,yes
9,>40,medium,yes,fair,yes


In [9]:
x = df.iloc[:, 0:4]
y = df['buys_computer']

x

,age,income,student,credit_rating
0,<=30,high,no,fair
1,<=30,high,no,excellent
2,31-40,high,no,fair
3,>40,medium,no,fair
4,>40,low,yes,fair
5,>40,low,yes,excellent
6,31-40,low,yes,excellent
7,<=30,medium,no,fair
8,<=30,low,yes,fair
9,>40,medium,yes,fair


In [11]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=1
)

df

,age,income,student,credit_rating,buys_computer
0,<=30,high,no,fair,no
1,<=30,high,no,excellent,no
2,31-40,high,no,fair,yes
3,>40,medium,no,fair,yes
4,>40,low,yes,fair,yes
5,>40,low,yes,excellent,no
6,31-40,low,yes,excellent,yes
7,<=30,medium,no,fair,no
8,<=30,low,yes,fair,yes
9,>40,medium,yes,fair,yes


In [15]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Reset train and test splits to ensure clean string columns before encoding
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=1
)

oe_age = OrdinalEncoder(categories=[['<=30', '31-40', '>40']])
oe_income = OrdinalEncoder(categories=[['low', 'medium', 'high']])
oe_credit = OrdinalEncoder(categories=[['fair', 'excellent']])

x_train['age'] = oe_age.fit_transform(x_train[['age']])
x_train['income'] = oe_income.fit_transform(x_train[['income']])
x_train['credit_rating'] = oe_credit.fit_transform(x_train[['credit_rating']])

x_test['age'] = oe_age.transform(x_test[['age']])
x_test['income'] = oe_income.transform(x_test[['income']])
x_test['credit_rating'] = oe_credit.transform(x_test[['credit_rating']])

###### One Hot Encoder on student column ######
ohe = OneHotEncoder(drop='if_binary', sparse_output=False)

x_train_new = ohe.fit_transform(x_train[['student']])
x_test_new = ohe.transform(x_test[['student']])

x_train = x_train.drop('student', axis=1)
x_test = x_test.drop('student', axis=1)

x_train_new

array([[0.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [1.]])

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)


x_train

,age,income,credit_rating
2,1.0,2.0,0.0
10,0.0,1.0,1.0
4,2.0,0.0,0.0
1,0.0,2.0,1.0
12,1.0,2.0,0.0
0,0.0,2.0,0.0
13,2.0,1.0,1.0
9,2.0,1.0,0.0
8,0.0,0.0,0.0
11,1.0,1.0,1.0


In [21]:
from sklearn.naive_bayes import CategoricalNB

model = CategoricalNB()
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
y_pred

df

,age,income,student,credit_rating,buys_computer
0,<=30,high,no,fair,no
1,<=30,high,no,excellent,no
2,31-40,high,no,fair,yes
3,>40,medium,no,fair,yes
4,>40,low,yes,fair,yes
5,>40,low,yes,excellent,no
6,31-40,low,yes,excellent,yes
7,<=30,medium,no,fair,no
8,<=30,low,yes,fair,yes
9,>40,medium,yes,fair,yes


In [22]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:3%}")

Accuracy: 66.666667%
